In [75]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import CLIPProcessor, CLIPModel
from tqdm.auto import tqdm

In [54]:
output_path = "../Output/few-shot-main-result.csv"

df = pd.read_csv(output_path)

print(df.shape)
print(df.columns)
df.head()

(2000, 14)
Index(['row_id', 'image_id', 'image_path', 'text', 'true_label',
       'sarcasm_rationale', 'not_sarcasm_rationale', 'top_sarcasm_ids',
       'top_sarcasm_texts', 'top_sarcasm_scores', 'top_non_sarcasm_ids',
       'top_non_sarcasm_texts', 'top_non_sarcasm_scores', 'error'],
      dtype='object')


,row_id,image_id,image_path,text,true_label,sarcasm_rationale,not_sarcasm_rationale,top_sarcasm_ids,top_sarcasm_texts,top_sarcasm_scores,top_non_sarcasm_ids,top_non_sarcasm_texts,top_non_sarcasm_scores,error
0,4181,819692441481711617,..\Dataset\dataset_image\819692441481711617.jpg,i 'm rethinking everything now,0,"The phrase ""I'm rethinking everything now"" pai...",The post expresses a thoughtful and sincere re...,"['914592921814433794', '921202878571864065']","['you ever think about that ? no , you only th...","[0.744874119758606, 0.7275177240371704]","['819694428277379074', '818607947706200064']","[""i 'm rethinking everything now"", 'wayment !']","[1.0000001192092896, 0.9519281983375549]",NaN
1,5418,923000568997539840,..\Dataset\dataset_image\923000568997539840.jpg,# truth about,1,The post uses exaggeration and ironic phrasing...,The text presents a straightforward sentiment ...,"['712358912264110080', '694584307655012358']","['true .', '# truth']","[0.7744077444076538, 0.7734344601631165]","['822225305779785730', '819694201831096321']","['exactly !', 'but really though']","[0.7239524722099304, 0.7229138016700745]",NaN
2,1850,818238963156680705,..\Dataset\dataset_image\818238963156680705.jpg,took a nap and slept through a pretty sunset i...,0,The post sarcastically contrasts the disappoin...,The post expresses a straightforward sentiment...,"['839491693049171968', '698338846287785984']","['haha when you know people care ! .... jk', ""...","[0.6213423013687134, 0.5821914672851562]","['822227785137655809', '822585733273808899']",['first thing i see on snap is food <user> <us...,"[0.7123841643333435, 0.6349201202392578]",NaN
3,5650,877881532404379648,..\Dataset\dataset_image\877881532404379648.jpg,- akhileshyadav is struggling for a govt job r...,1,The post sarcastically suggests that Akhilesh ...,The post straightforwardly describes Akhilesh ...,"['880436067270356993', '877715414763130880']",[': a govt employee got delighted after rescui...,"[0.7288748025894165, 0.7151439189910889]","['819332300399779841', '818238095594291200']",['queen of flowers queen of farmers priyanka g...,"[0.5446099638938904, 0.5340462923049927]",NaN
4,9656,822594021897961472,..\Dataset\dataset_image\822594021897961472.jpg,quand jma fait signer des cracks > > > > >,0,The post text suggests a boastful claim about ...,The post expresses pride and confidence in hav...,"['686926157288202241', '902261356774207489']",['im in the process of sealing a £ 12m switch ...,"[0.5847525596618652, 0.5751825571060181]","['819688464740450304', '822587074209583104']","['i have officially completed life .', 'brace ...","[0.6404160261154175, 0.6285971999168396]",NaN


In [55]:
IMAGE_DATA_DIR = "../Dataset/dataset_image"
DEVICE = 'cpu'
OUTPUT_DIR = Path("../Output")
CLIP_MODEL_ID = "openai/clip-vit-large-patch14"

In [56]:
IMAGE_DATA_DIR = Path(IMAGE_DATA_DIR)

In [57]:
processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID)

clip_model = clip_model.to(DEVICE)
clip_model.eval()

print("CLIP loaded:", CLIP_MODEL_ID)

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 8143.93it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded: openai/clip-vit-large-patch14


In [58]:
label_col = "true_label" if "true_label" in df.columns else "label"
required_cols = [
    "text",
    "sarcasm_rationale",
    "not_sarcasm_rationale",
    label_col
]

missing_rows = df[df[required_cols].isna().any(axis=1)]

print("Missing rows:", len(missing_rows))
display(missing_rows)

Missing rows: 0


,row_id,image_id,image_path,text,true_label,sarcasm_rationale,not_sarcasm_rationale,top_sarcasm_ids,top_sarcasm_texts,top_sarcasm_scores,top_non_sarcasm_ids,top_non_sarcasm_texts,top_non_sarcasm_scores,error


In [59]:
df = df.dropna(subset=[
    "text",
    "sarcasm_rationale",
    "not_sarcasm_rationale",
    label_col
]).reset_index(drop=True)

df[label_col] = df[label_col].astype(int)

print(df[label_col].value_counts())

true_label
0    1000
1    1000
Name: count, dtype: int64


In [60]:
label_col = "true_label"

train_cls_df, val_cls_df = train_test_split(
    df,
    test_size=0.2,
    random_state=67,
    stratify=df[label_col]
)

train_cls_df = train_cls_df.reset_index(drop=True)
val_cls_df = val_cls_df.reset_index(drop=True)

print("Train:")
print(train_cls_df[label_col].value_counts())

print("Val:")
print(val_cls_df[label_col].value_counts())

Train:
true_label
0    800
1    800
Name: count, dtype: int64
Val:
true_label
1    200
0    200
Name: count, dtype: int64


In [61]:
def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).replace("\n", " ").strip()


def get_image_path_from_row(row):
    if "image_id" in row and pd.notna(row["image_id"]):
        return IMAGE_DATA_DIR / f"{str(row['image_id'])}.jpg"

    raw_path = str(row["image_path"]).replace("\\", "/")
    filename = Path(raw_path).name
    return IMAGE_DATA_DIR / filename

In [62]:
class CofiParaRAGDataset(Dataset):
    def __init__(
        self,
        df,
        clip_processor,
        rationale_tokenizer,
        label_col="true_label",
        max_query_text_len=77,
        max_rationale_len=120
    ):
        self.df = df.reset_index(drop=True)
        self.clip_processor = clip_processor
        self.rationale_tokenizer = rationale_tokenizer
        self.label_col = label_col
        self.max_query_text_len = max_query_text_len
        self.max_rationale_len = max_rationale_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = get_image_path_from_row(row)
        image = Image.open(image_path).convert("RGB")

        image_inputs = self.clip_processor(
            images=image,
            return_tensors="pt"
        )

        query_text_inputs = self.clip_processor(
            text=[safe_text(row["text"])],
            return_tensors="pt",
            padding="max_length",
            max_length=self.max_query_text_len,
            truncation=True
        )

        pos_inputs = self.rationale_tokenizer(
            safe_text(row["sarcasm_rationale"]),
            return_tensors="pt",
            padding="max_length",
            max_length=self.max_rationale_len,
            truncation=True
        )

        neg_inputs = self.rationale_tokenizer(
            safe_text(row["not_sarcasm_rationale"]),
            return_tensors="pt",
            padding="max_length",
            max_length=self.max_rationale_len,
            truncation=True
        )

        return {
            "pixel_values": image_inputs["pixel_values"].squeeze(0),

            "query_input_ids": query_text_inputs["input_ids"].squeeze(0),
            "query_attention_mask": query_text_inputs["attention_mask"].squeeze(0),

            "pos_input_ids": pos_inputs["input_ids"].squeeze(0),
            "pos_attention_mask": pos_inputs["attention_mask"].squeeze(0),

            "neg_input_ids": neg_inputs["input_ids"].squeeze(0),
            "neg_attention_mask": neg_inputs["attention_mask"].squeeze(0),

            "label": torch.tensor(int(row[self.label_col]), dtype=torch.long),
        }

In [63]:
rationale_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
roberta_model = AutoModel.from_pretrained("roberta-base")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7253.40it/s]
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [64]:
class CrossAttentionFusion(nn.Module):
    def __init__(self, dim=768, num_heads=8, dropout=0.1):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.Dropout(dropout)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, query_tokens, key_value_tokens, key_value_mask=None):
        key_padding_mask = None

        if key_value_mask is not None:
            key_padding_mask = key_value_mask == 0

        attn_out, _ = self.attn(
            query=query_tokens,
            key=key_value_tokens,
            value=key_value_tokens,
            key_padding_mask=key_padding_mask,
            need_weights=False
        )

        x = self.norm1(query_tokens + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))

        return x

In [65]:
def masked_mean_pool(tokens, mask):
    mask = mask.unsqueeze(-1).float()
    return (tokens * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

In [66]:
class CofiParaRAGClassifier(nn.Module):
    def __init__(
        self,
        clip_model,
        roberta_model,
        dim=768,
        num_heads=8,
        dropout=0.1,
        freeze_clip=True,
        freeze_roberta=False,
        num_classes=2
    ):
        super().__init__()

        self.clip_model = clip_model
        self.roberta_model = roberta_model
        self.freeze_clip = freeze_clip
        self.freeze_roberta = freeze_roberta

        if freeze_clip:
            for p in self.clip_model.parameters():
                p.requires_grad = False

        if freeze_roberta:
            for p in self.roberta_model.parameters():
                p.requires_grad = False

        self.image_text_fusion = CrossAttentionFusion(
            dim=dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.pos_fusion = CrossAttentionFusion(
            dim=dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.neg_fusion = CrossAttentionFusion(
            dim=dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.classifier = nn.Sequential(
            nn.Linear(dim * 2, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def encode_image_tokens(self, pixel_values):
        vision_outputs = self.clip_model.vision_model(
            pixel_values=pixel_values,
            return_dict=True
        )

        image_tokens = vision_outputs.last_hidden_state

        image_tokens = self.clip_model.vision_model.post_layernorm(image_tokens)
        image_tokens = self.clip_model.visual_projection(image_tokens)
        image_tokens = F.normalize(image_tokens, p=2, dim=-1)

        return image_tokens

    def encode_query_text_tokens(self, input_ids, attention_mask):
        text_outputs = self.clip_model.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        text_tokens = text_outputs.last_hidden_state
        text_tokens = self.clip_model.text_projection(text_tokens)
        text_tokens = F.normalize(text_tokens, p=2, dim=-1)

        return text_tokens

    def encode_rationale_tokens(self, input_ids, attention_mask):
        outputs = self.roberta_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        return outputs.last_hidden_state

    def forward(
        self,
        pixel_values,
        query_input_ids,
        query_attention_mask,
        pos_input_ids,
        pos_attention_mask,
        neg_input_ids,
        neg_attention_mask
    ):
        if self.freeze_clip:
            with torch.no_grad():
                image_tokens = self.encode_image_tokens(pixel_values)
                query_text_tokens = self.encode_query_text_tokens(
                    query_input_ids,
                    query_attention_mask
                )
        else:
            image_tokens = self.encode_image_tokens(pixel_values)
            query_text_tokens = self.encode_query_text_tokens(
                query_input_ids,
                query_attention_mask
            )

        if self.freeze_roberta:
            with torch.no_grad():
                pos_tokens = self.encode_rationale_tokens(
                    pos_input_ids,
                    pos_attention_mask
                )
                neg_tokens = self.encode_rationale_tokens(
                    neg_input_ids,
                    neg_attention_mask
                )
        else:
            pos_tokens = self.encode_rationale_tokens(
                pos_input_ids,
                pos_attention_mask
            )
            neg_tokens = self.encode_rationale_tokens(
                neg_input_ids,
                neg_attention_mask
            )

        multimodal_tokens = self.image_text_fusion(
            query_tokens=query_text_tokens,
            key_value_tokens=image_tokens,
            key_value_mask=None
        )

        pos_fused = self.pos_fusion(
            query_tokens=multimodal_tokens,
            key_value_tokens=pos_tokens,
            key_value_mask=pos_attention_mask
        )

        neg_fused = self.neg_fusion(
            query_tokens=multimodal_tokens,
            key_value_tokens=neg_tokens,
            key_value_mask=neg_attention_mask
        )

        concat_tokens = torch.cat([pos_fused, neg_fused], dim=-1)

        pooled = masked_mean_pool(concat_tokens, query_attention_mask)

        logits = self.classifier(pooled)

        return logits

In [67]:
BATCH_SIZE = 2

train_dataset = CofiParaRAGDataset(
    df=train_cls_df,
    clip_processor=processor,
    rationale_tokenizer=rationale_tokenizer,
    label_col=label_col
)

val_dataset = CofiParaRAGDataset(
    df=val_cls_df,
    clip_processor=processor,
    rationale_tokenizer=rationale_tokenizer,
    label_col=label_col
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [68]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classifier_model = CofiParaRAGClassifier(
    clip_model=clip_model,
    roberta_model=roberta_model,
    dim=768,
    num_heads=8,
    dropout=0.1,
    freeze_clip=True,
    freeze_roberta=False,
    num_classes=2
).to(DEVICE)

In [69]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, classifier_model.parameters()),
    lr=2e-5,
    weight_decay=0.01
)

criterion = nn.CrossEntropyLoss()

In [70]:
def move_batch_to_device(batch, device):
    return {
        k: v.to(device)
        for k, v in batch.items()
        if torch.is_tensor(v)
    }

In [76]:
def train_one_epoch_verbose(model, dataloader, optimizer, criterion, device, epoch=None, total_epochs=None):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    desc = "Training"
    if epoch is not None and total_epochs is not None:
        desc = f"Training Epoch {epoch}/{total_epochs}"

    progress_bar = tqdm(dataloader, desc=desc, leave=True)

    for step, batch in enumerate(progress_bar, start=1):
        b = move_batch_to_device(batch, device)

        optimizer.zero_grad()

        logits = model(
            pixel_values=b["pixel_values"],
            query_input_ids=b["query_input_ids"],
            query_attention_mask=b["query_attention_mask"],
            pos_input_ids=b["pos_input_ids"],
            pos_attention_mask=b["pos_attention_mask"],
            neg_input_ids=b["neg_input_ids"],
            neg_attention_mask=b["neg_attention_mask"]
        )

        loss = criterion(logits, b["label"])

        loss.backward()
        optimizer.step()

        batch_size = b["label"].size(0)
        total_loss += loss.item() * batch_size

        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(b["label"].detach().cpu().numpy().tolist())

        running_loss = total_loss / len(all_labels)
        running_acc = accuracy_score(all_labels, all_preds)

        progress_bar.set_postfix({
            "loss": f"{running_loss:.4f}",
            "acc": f"{running_acc:.4f}",
            "batch_loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, macro_f1

In [77]:
@torch.no_grad()
def evaluate_verbose(model, dataloader, criterion, device, epoch=None, total_epochs=None):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    desc = "Validation"
    if epoch is not None and total_epochs is not None:
        desc = f"Validation Epoch {epoch}/{total_epochs}"

    progress_bar = tqdm(dataloader, desc=desc, leave=True)

    for step, batch in enumerate(progress_bar, start=1):
        b = move_batch_to_device(batch, device)

        logits = model(
            pixel_values=b["pixel_values"],
            query_input_ids=b["query_input_ids"],
            query_attention_mask=b["query_attention_mask"],
            pos_input_ids=b["pos_input_ids"],
            pos_attention_mask=b["pos_attention_mask"],
            neg_input_ids=b["neg_input_ids"],
            neg_attention_mask=b["neg_attention_mask"]
        )

        loss = criterion(logits, b["label"])

        batch_size = b["label"].size(0)
        total_loss += loss.item() * batch_size

        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(b["label"].detach().cpu().numpy().tolist())

        running_loss = total_loss / len(all_labels)
        running_acc = accuracy_score(all_labels, all_preds)

        progress_bar.set_postfix({
            "loss": f"{running_loss:.4f}",
            "acc": f"{running_acc:.4f}",
            "batch_loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, macro_f1, all_labels, all_preds

In [78]:
EPOCHS = 5

best_val_f1 = 0.0
best_path = OUTPUT_DIR / "cofipara_rag_classifier.pt"

history = []

for epoch in range(1, EPOCHS + 1):
    print("=" * 100)
    print(f"Epoch {epoch}/{EPOCHS}")
    print("=" * 100)

    train_loss, train_acc, train_f1 = train_one_epoch_verbose(
        classifier_model,
        train_loader,
        optimizer,
        criterion,
        DEVICE,
        epoch=epoch,
        total_epochs=EPOCHS
    )

    val_loss, val_acc, val_f1, y_true, y_pred = evaluate_verbose(
        classifier_model,
        val_loader,
        criterion,
        DEVICE,
        epoch=epoch,
        total_epochs=EPOCHS
    )

    epoch_result = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_macro_f1": train_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_macro_f1": val_f1,
    }

    history.append(epoch_result)

    print("\nEpoch Summary")
    print(f"Train loss     : {train_loss:.4f}")
    print(f"Train acc      : {train_acc:.4f}")
    print(f"Train macro F1 : {train_f1:.4f}")
    print(f"Val loss       : {val_loss:.4f}")
    print(f"Val acc        : {val_acc:.4f}")
    print(f"Val macro F1   : {val_f1:.4f}")

    print("\nValidation classification report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=["not sarcasm", "sarcasm"],
        digits=4
    ))

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(classifier_model.state_dict(), best_path)
        print(f"Saved best model to {best_path} with val_macro_f1={best_val_f1:.4f}")
    else:
        print(f"No improvement. Best val_macro_f1 still {best_val_f1:.4f}")

    print()

Epoch 1/5


Validation Epoch 1/5: 100%|██████████| 200/200 [03:52<00:00,  1.16s/it, loss=0.4697, acc=0.7875, batch_loss=0.2446]



Epoch Summary
Train loss     : 0.5545
Train acc      : 0.7100
Train macro F1 : 0.7098
Val loss       : 0.4697
Val acc        : 0.7875
Val macro F1   : 0.7857

Validation classification report:
              precision    recall  f1-score   support

 not sarcasm     0.8528    0.6950    0.7658       200
     sarcasm     0.7426    0.8800    0.8055       200

    accuracy                         0.7875       400
   macro avg     0.7977    0.7875    0.7857       400
weighted avg     0.7977    0.7875    0.7857       400

Saved best model to ..\Output\cofipara_rag_classifier.pt with val_macro_f1=0.7857

Epoch 2/5


Validation Epoch 2/5: 100%|██████████| 200/200 [03:46<00:00,  1.13s/it, loss=0.4441, acc=0.8075, batch_loss=0.0666]



Epoch Summary
Train loss     : 0.3847
Train acc      : 0.8244
Train macro F1 : 0.8242
Val loss       : 0.4441
Val acc        : 0.8075
Val macro F1   : 0.8075

Validation classification report:
              precision    recall  f1-score   support

 not sarcasm     0.8122    0.8000    0.8060       200
     sarcasm     0.8030    0.8150    0.8089       200

    accuracy                         0.8075       400
   macro avg     0.8076    0.8075    0.8075       400
weighted avg     0.8076    0.8075    0.8075       400

Saved best model to ..\Output\cofipara_rag_classifier.pt with val_macro_f1=0.8075

Epoch 3/5


...


Epoch Summary
Train loss     : 0.2982
Train acc      : 0.8706
Train macro F1 : 0.8706
Val loss       : 0.4315
Val acc        : 0.8025
Val macro F1   : 0.8025

Validation classification report:
              precision    recall  f1-score   support

 not sarcasm     0.8103    0.7900    0.8000       200
     sarcasm     0.7951    0.8150    0.8049       200

    accuracy                         0.8025       400
   macro avg     0.8027    0.8025    0.8025       400
weighted avg     0.8027    0.8025    0.8025       400

No improvement. Best val_macro_f1 still 0.8075

Epoch 4/5


Validation Epoch 4/5: 100%|██████████| 200/200 [03:54<00:00,  1.17s/it, loss=0.6452, acc=0.7800, batch_loss=0.0057]



Epoch Summary
Train loss     : 0.2545
Train acc      : 0.8981
Train macro F1 : 0.8981
Val loss       : 0.6452
Val acc        : 0.7800
Val macro F1   : 0.7753

Validation classification report:
              precision    recall  f1-score   support

 not sarcasm     0.7171    0.9250    0.8079       200
     sarcasm     0.8944    0.6350    0.7427       200

    accuracy                         0.7800       400
   macro avg     0.8057    0.7800    0.7753       400
weighted avg     0.8057    0.7800    0.7753       400

No improvement. Best val_macro_f1 still 0.8075

Epoch 5/5


Validation Epoch 5/5: 100%|██████████| 200/200 [03:43<00:00,  1.12s/it, loss=0.5409, acc=0.8050, batch_loss=0.0133]


Epoch Summary
Train loss     : 0.1956
Train acc      : 0.9206
Train macro F1 : 0.9206
Val loss       : 0.5409
Val acc        : 0.8050
Val macro F1   : 0.8047

Validation classification report:
              precision    recall  f1-score   support

 not sarcasm     0.8315    0.7650    0.7969       200
     sarcasm     0.7824    0.8450    0.8125       200

    accuracy                         0.8050       400
   macro avg     0.8070    0.8050    0.8047       400
weighted avg     0.8070    0.8050    0.8047       400

No improvement. Best val_macro_f1 still 0.8075



In [79]:
history_df = pd.DataFrame(history)
history_path = OUTPUT_DIR / "cofipara_rag_classifier_history.csv"

history_df.to_csv(history_path, index=False)

print("Saved history to:", history_path)
display(history_df)

Saved history to: ..\Output\cofipara_rag_classifier_history.csv


,epoch,train_loss,train_acc,train_macro_f1,val_loss,val_acc,val_macro_f1
0,1,0.554513,0.710000,0.709819,0.469673,0.7875,0.785666
1,2,0.384748,0.824375,0.824210,0.444140,0.8075,0.807489
2,3,0.298173,0.870625,0.870616,0.431478,0.8025,0.802469
3,4,0.254470,0.898125,0.898104,0.645202,0.7800,0.775275
4,5,0.195602,0.920625,0.920614,0.540853,0.8050,0.804688


In [81]:
classifier_model.load_state_dict(torch.load(best_path, map_location=DEVICE))

val_loss, val_acc, val_f1, y_true, y_pred = evaluate_verbose(
    classifier_model,
    val_loader,
    criterion,
    DEVICE
)

print("Val loss:", val_loss)
print("Val acc:", val_acc)
print("Val macro F1:", val_f1)

print(classification_report(
    y_true,
    y_pred,
    target_names=["not sarcasm", "sarcasm"],
    digits=4
))

Validation: 100%|██████████| 200/200 [06:29<00:00,  1.95s/it, loss=0.4441, acc=0.8075, batch_loss=0.0666]

Val loss: 0.44414036453235894
Val acc: 0.8075
Val macro F1: 0.8074891712658837
              precision    recall  f1-score   support

 not sarcasm     0.8122    0.8000    0.8060       200
     sarcasm     0.8030    0.8150    0.8089       200

    accuracy                         0.8075       400
   macro avg     0.8076    0.8075    0.8075       400
weighted avg     0.8076    0.8075    0.8075       400

